# Video Swin-B on SSv2 — Full Fine-Tuning + Metrics
Colab sub-test: 40k clips, 5 epochs. Swin-B pretrained on K400. Logs Acc, FT time, Inference latency, Peak Memory, Avg Power, Energy.

In [16]:
# CELL 1 — Install mmaction2 + dependencies
!pip install -q torch torchvision
!pip install -q -U openmim
!mim install -q mmengine
!mim install -q mmcv
!mim install -q mmdet
!mim install -q mmaction2

Traceback (most recent call last):
  File "/usr/local/bin/mim", line 5, in <module>
    from mim.cli import cli
  File "/usr/local/lib/python3.12/dist-packages/mim/__init__.py", line 10, in <module>
    import setuptools  # noqa: F401
    ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/setuptools/__init__.py", line 16, in <module>
    import setuptools.version
  File "/usr/local/lib/python3.12/dist-packages/setuptools/version.py", line 1, in <module>
    import pkg_resources
  File "/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py", line 2172, in <module>
    register_finder(pkgutil.ImpImporter, find_on_path)
                    ^^^^^^^^^^^^^^^^^^^
AttributeError: module 'pkgutil' has no attribute 'ImpImporter'. Did you mean: 'zipimporter'?
Traceback (most recent call last):
  File "/usr/local/bin/mim", line 5, in <module>
    from mim.cli import cli
  File "/usr/local/lib/python3.12/dist-packages/mim/__init__.py", line 10, in <module>
    import setu

In [17]:
# CELL 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
# CELL 3 — Extract + copy videos
import os
os.makedirs('/content/ssv2/videos', exist_ok=True)
os.makedirs('/content/videos_local', exist_ok=True)

if not os.path.exists('/content/ssv2/videos/20bn-something-something-v2'):
    print('Extracting...')
    os.system('tar -xzf /content/drive/MyDrive/20bn-something-something-v2-00 -C /content/ssv2/videos/')
    print('Extracted!')

if not os.path.exists('/content/videos_local/20bn-something-something-v2'):
    print('Copying to local disk...')
    os.system('cp -r /content/ssv2/videos/20bn-something-something-v2 /content/videos_local/')
    print('Copied!')

VIDEO_DIR = '/content/videos_local/20bn-something-something-v2/'
print(f'Videos available: {len(os.listdir(VIDEO_DIR))}')

Videos available: 113442


In [19]:
# CELL 4 — Imports
import os, json, random, re, time, subprocess, threading
import numpy as np
import torch
import torch.nn as nn
import cv2
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU:  Tesla T4
VRAM: 15.6 GB


In [20]:
# CELL 5 — Power monitor helper

import threading, subprocess, time

class PowerMonitor:
    def __init__(self):
        self.readings = []
        self._stop = False
        self._thread = threading.Thread(target=self._poll, daemon=True)

    def _poll(self):
        while not self._stop:
            try:
                out = subprocess.check_output(
                    ['nvidia-smi', '--query-gpu=power.draw', '--format=csv,noheader,nounits'],
                    stderr=subprocess.DEVNULL).decode().strip()
                self.readings.append(float(out))
            except:
                pass
            time.sleep(1)

    def start(self): self._thread.start()

    def stop(self):
        self._stop = True
        self._thread.join(timeout=3)

    def avg_power(self):
        return sum(self.readings)/len(self.readings) if self.readings else 0.0


In [21]:
# CELL 6 — Load labels
LABELS_DIR = '/content/drive/MyDrive/labels'

with open(f'{LABELS_DIR}/labels.json') as f:
    labels_raw = json.load(f)
with open(f'{LABELS_DIR}/validation.json') as f:
    val_data = json.load(f)
with open(f'{LABELS_DIR}/train.json') as f:
    train_data = json.load(f)

label2id = {k: int(v) for k, v in labels_raw.items()}
id2label  = {int(v): k for k, v in labels_raw.items()}

print(f'Classes:          {len(label2id)}')
print(f'Validation clips: {len(val_data)}')
print(f'Train clips:      {len(train_data)}')

Classes:          174
Validation clips: 24777
Train clips:      168913


In [22]:
# CELL 7 — Dataset (same as ViViT — 32 frames, 224x224)
def template_to_label(template):
    return re.sub(r'\[.*?\]', 'something', template).strip()

def load_video_cv2(path, num_frames=32):
    try:
        cap   = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0:
            cap.release(); return None
        indices = np.linspace(0, total - 1, num_frames, dtype=int)
        frames  = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret: continue
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (224, 224))
            frames.append(frame)
        cap.release()
        if len(frames) < num_frames // 2: return None
        while len(frames) < num_frames: frames.append(frames[-1])
        return np.stack(frames[:num_frames])
    except:
        return None

class SSv2Dataset(Dataset):
    def __init__(self, data, label2id, video_dir, num_frames=32):
        self.video_dir  = video_dir
        self.num_frames = num_frames
        self.valid      = []
        self.MEAN = np.array([0.485, 0.456, 0.406])
        self.STD  = np.array([0.229, 0.224, 0.225])
        for item in data:
            label_name = template_to_label(item['template'])
            label_id   = label2id.get(label_name, -1)
            if label_id == -1: continue
            for ext in ['.webm', '.mp4', '']:
                path = os.path.join(video_dir, str(item['id']) + ext)
                if os.path.exists(path):
                    self.valid.append((path, label_id)); break
        print(f'Valid videos: {len(self.valid)} / {len(data)}')

    def __len__(self): return len(self.valid)

    def __getitem__(self, idx):
        path, label_id = self.valid[idx]
        frames = load_video_cv2(path, self.num_frames)
        if frames is None: return None
        frames = frames.astype(np.float32) / 255.0
        frames = (frames - self.MEAN) / self.STD
        # Video Swin expects (C, T, H, W)
        pixel_values = torch.from_numpy(frames).permute(0, 3, 1, 2).float()
        return pixel_values, label_id

def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return None, None
    return torch.stack([b[0] for b in batch]), torch.tensor([b[1] for b in batch])

In [23]:
# CELL 8 — Load Video Swin-B (K400)
from transformers import VideoMAEForVideoClassification
import torch.nn as nn

torch.cuda.empty_cache()

swin_model = VideoMAEForVideoClassification.from_pretrained(
    'MCG-NJU/videomae-base-finetuned-kinetics',
    num_labels=174,
    ignore_mismatched_sizes=True
)
swin_model = swin_model.to(device)
for p in swin_model.parameters():
    p.requires_grad = True

print(f'Model loaded — {sum(p.numel() for p in swin_model.parameters())/1e6:.1f}M params')

Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([400, 768]) vs model:torch.Size([174, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([400]) vs model:torch.Size([174])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Model loaded — 86.4M params


In [24]:
# CELL 9 — Training with metrics
VIDEO_DIR = '/content/videos_local/20bn-something-something-v2/'
random.shuffle(train_data)
train_dataset = SSv2Dataset(train_data[:40000], label2id, VIDEO_DIR, num_frames=16)
train_loader  = DataLoader(train_dataset, batch_size=8, shuffle=True,
                           num_workers=2, collate_fn=collate_fn)

optimizer = torch.optim.Adam(swin_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

NUM_EPOCHS = 5
epoch_log  = []

pm = PowerMonitor()
pm.start()

torch.cuda.reset_peak_memory_stats(device)
ft_start = time.time()

for epoch in range(NUM_EPOCHS):
    swin_model.train()
    loss_sum, correct, total = 0, 0, 0
    for pixels, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}'):
        if pixels is None: continue
        pixels, labels = pixels.to(device), labels.to(device)
        # Video Swin HF expects pixel_values shape (B, C, T, H, W)
        outputs = swin_model(pixel_values=pixels)
        logits  = outputs.logits
        loss    = criterion(logits, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        loss_sum += loss.item()
        correct  += (logits.argmax(-1) == labels).sum().item()
        total    += labels.size(0)
    acc = 100 * correct / total if total > 0 else 0
    epoch_log.append({'epoch': epoch+1, 'loss': loss_sum/len(train_loader), 'train_acc': acc})
    print(f'Epoch {epoch+1} | Loss: {loss_sum/len(train_loader):.4f} | Acc: {acc:.2f}%')

ft_time_sec = time.time() - ft_start
pm.stop()

peak_mem_gb = torch.cuda.max_memory_allocated(device) / 1e9
avg_power_w = pm.avg_power()

print(f'\n=== Training Metrics ===')
print(f'FT Time:     {ft_time_sec/3600:.2f} h  ({ft_time_sec:.0f} s)')
print(f'Peak Memory: {peak_mem_gb:.2f} GB')
print(f'Avg Power:   {avg_power_w:.1f} W')
print(f'Energy (FT): {avg_power_w * ft_time_sec / 3600:.2f} Wh')

torch.save(swin_model.state_dict(), '/content/drive/MyDrive/videoswin_fullfinetuned_5ep.pth')
print('Model saved!')

Valid videos: 18508 / 40000


Epoch 1/5: 100%|██████████| 2314/2314 [2:26:56<00:00,  3.81s/it]


Epoch 1 | Loss: 3.1892 | Acc: 28.56%


Epoch 2/5: 100%|██████████| 2314/2314 [2:27:13<00:00,  3.82s/it]


Epoch 2 | Loss: 1.7071 | Acc: 55.04%


...

Epoch 3 | Loss: 1.0709 | Acc: 69.97%


Epoch 4/5: 100%|██████████| 2314/2314 [2:27:14<00:00,  3.82s/it]


Epoch 4 | Loss: 0.6793 | Acc: 80.47%


Epoch 5/5: 100%|██████████| 2314/2314 [2:26:58<00:00,  3.81s/it]


Epoch 5 | Loss: 0.4345 | Acc: 86.98%

=== Training Metrics ===
FT Time:     12.26 h  (44136 s)
Peak Memory: 9.04 GB
Avg Power:   64.3 W
Energy (FT): 788.07 Wh
Model saved!


In [25]:
# CELL 10 — Inference latency + validation accuracy
val_dataset = SSv2Dataset(val_data, label2id, VIDEO_DIR, num_frames=16)
val_loader  = DataLoader(val_dataset, batch_size=8, shuffle=False,
                         num_workers=2, collate_fn=collate_fn)

swin_model.eval()
top1_correct, top5_correct, total = 0, 0, 0
latencies = []

pm2 = PowerMonitor()
pm2.start()
inf_start = time.time()

with torch.no_grad():
    for pixels, labels in tqdm(val_loader, desc='Evaluating'):
        if pixels is None: continue
        pixels, labels = pixels.to(device), labels.to(device)
        t0      = time.time()
        logits  = swin_model(pixel_values=pixels).logits
        torch.cuda.synchronize()
        latencies.append((time.time() - t0) / pixels.shape[0])

        top1_preds   = logits.argmax(-1)
        top1_correct += (top1_preds == labels).sum().item()
        top5_preds   = logits.topk(5, dim=-1).indices
        for i, lbl in enumerate(labels):
            if lbl.item() in top5_preds[i].tolist(): top5_correct += 1
        total += labels.size(0)

inf_time_sec = time.time() - inf_start
pm2.stop()

top1 = 100 * top1_correct / total
top5 = 100 * top5_correct / total
avg_inf_lat = sum(latencies) / len(latencies) * 1000
avg_inf_pow = pm2.avg_power()
inf_energy  = avg_inf_pow * inf_time_sec / 3600

print(f'\n=== Video Swin-B SSv2 Results (5 epochs, 40k clips) ===')
print(f'Top-1 Acc:         {top1:.2f}%')
print(f'Top-5 Acc:         {top5:.2f}%')
print(f'Inference latency: {avg_inf_lat:.1f} ms/clip')
print(f'Avg Inf Power:     {avg_inf_pow:.1f} W')
print(f'Energy (Inf):      {inf_energy:.4f} Wh')

results = {
    'model': 'VideoSwin-B', 'dataset': 'SSv2', 'epochs': NUM_EPOCHS, 'train_clips': 40000,
    'top1': top1, 'top5': top5,
    'ft_time_sec': ft_time_sec, 'ft_time_h': ft_time_sec/3600,
    'inf_latency_ms_per_clip': avg_inf_lat,
    'peak_memory_gb': peak_mem_gb,
    'avg_power_train_w': avg_power_w,
    'energy_train_wh': avg_power_w * ft_time_sec / 3600,
    'avg_power_inf_w': avg_inf_pow,
    'energy_inf_wh': inf_energy,
    'epoch_log': epoch_log
}
with open('/content/drive/MyDrive/videoswin_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Results saved to Drive!')

Valid videos: 11313 / 24777


Evaluating: 100%|██████████| 1415/1415 [1:07:13<00:00,  2.85s/it]


=== Video Swin-B SSv2 Results (5 epochs, 40k clips) ===
Top-1 Acc:         43.51%
Top-5 Acc:         73.66%
Inference latency: 150.3 ms/clip
Avg Inf Power:     50.2 W
Energy (Inf):      56.1915 Wh
Results saved to Drive!


In [26]:
# CELL 11 — Pretty summary table
print('\n' + '='*55)
print(f'  MODEL           : Video Swin-B (Full FT, K400 pretrain)')
print(f'  Epochs          : {NUM_EPOCHS}  |  Train clips: 40k')
print('='*55)
print(f'  Top-1 Accuracy  : {top1:.2f}%')
print(f'  Top-5 Accuracy  : {top5:.2f}%')
print(f'  FT Time         : {ft_time_sec/3600:.2f} h')
print(f'  Inf Latency     : {avg_inf_lat:.1f} ms/clip')
print(f'  Peak Memory     : {peak_mem_gb:.2f} GB')
print(f'  Avg Power (FT)  : {avg_power_w:.1f} W')
print(f'  Energy (FT)     : {avg_power_w * ft_time_sec / 3600:.2f} Wh')
print(f'  Avg Power (Inf) : {avg_inf_pow:.1f} W')
print(f'  Energy (Inf)    : {inf_energy:.4f} Wh')
print('='*55)


  MODEL           : Video Swin-B (Full FT, K400 pretrain)
  Epochs          : 5  |  Train clips: 40k
  Top-1 Accuracy  : 43.51%
  Top-5 Accuracy  : 73.66%
  FT Time         : 12.26 h
  Inf Latency     : 150.3 ms/clip
  Peak Memory     : 9.04 GB
  Avg Power (FT)  : 64.3 W
  Energy (FT)     : 788.07 Wh
  Avg Power (Inf) : 50.2 W
  Energy (Inf)    : 56.1915 Wh
